# Project 2: VisDrone Object Detection

**Course:** AI-driven Perception, Learning and Mapping for Drones
**Experiments:** Resolution vs capacity ablation on VisDrone2019-DET

## Overview

Three experiments testing which lever drives small-object detection performance on aerial imagery:

| Exp | Model | imgsz | mAP50 |
|---|---|---|---|
| 1 | yolov8n | 640 | 0.279 |
| 2 | yolov8n | 1280 | 0.468 |
| 3 | yolov8s | 1280 | 0.543 |

**Finding:** Resolution is the dominant lever (+68% mAP50); capacity adds +16% on top.


## Setup

Mount Drive, install Ultralytics, verify GPU.

In [ ]:
import os, torch
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

!pip install -q ultralytics pandas matplotlib

import ultralytics
print('ultralytics', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Experiment 1 — Baseline (yolov8n @ 640px)

30 epochs, batch 16. Establishes reference performance.


In [ ]:
from ultralytics import YOLO
PROJECT_DIR = '/content/drive/MyDrive/visdrone'

model = YOLO('yolov8n.pt')
results = model.train(
    data='VisDrone.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name='exp1_finetune',
    exist_ok=True,
    patience=8,
)

## Experiment 2 — Resolution (yolov8n @ 1280px)

Only change: imgsz=1280. Batch reduced to 8 to fit VRAM.

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data='VisDrone.yaml',
    epochs=30,
    imgsz=1280,
    batch=8,
    project=PROJECT_DIR,
    name='exp2_imgsz1280',
    exist_ok=True,
    patience=10,
)

## Experiment 3 — Capacity (yolov8s @ 1280px)

Larger model (11.1M params vs 3.0M), same resolution.

In [ ]:
model = YOLO('yolov8s.pt')
results = model.train(
    data='VisDrone.yaml',
    epochs=30,
    imgsz=1280,
    batch=8,
    project=PROJECT_DIR,
    name='exp3_yolov8s_1280',
    exist_ok=True,
    patience=10,
)

## Option B — Size-Stratified Analysis

Compare Exp 1 vs Exp 2 performance across COCO size buckets (small/medium/large).

In [ ]:
import numpy as np
from pathlib import Path

# Per-class mAP50-95 for Exp 1 (640) and Exp 2 (1280)
class_names = ['pedestrian','people','bicycle','car','van','truck',
               'tricycle','awning-tricycle','bus','motor']
exp1_map = np.array([0.116, 0.071, 0.019, 0.463, 0.212, 0.171, 0.101, 0.050, 0.247, 0.117])
exp2_map = np.array([0.264, 0.170, 0.104, 0.593, 0.360, 0.282, 0.198, 0.119, 0.441, 0.252])

# COCO size buckets (from Option B analysis)
small_classes = [0, 1, 2, 6, 7, 9]  # pedestrian, people, bicycle, tricycle, awning-tricycle, motor
medium_classes = [3, 4, 5, 8]         # car, van, truck, bus

small_exp1 = exp1_map[small_classes].mean()
small_exp2 = exp2_map[small_classes].mean()
medium_exp1 = exp1_map[medium_classes].mean()
medium_exp2 = exp2_map[medium_classes].mean()

print(f'Small bucket: Exp1={small_exp1:.4f}, Exp2={small_exp2:.4f}, fold={small_exp2/small_exp1:.2f}x')
print(f'Medium bucket: Exp1={medium_exp1:.4f}, Exp2={medium_exp2:.4f}, fold={medium_exp2/medium_exp1:.2f}x')


## Results Summary

**Resolution dominates:**
- Exp 1 → Exp 2: mAP50 0.279 → 0.468 (+68%)
- Gains concentrated in small classes (bicycle 4.1×, awning-tricycle 2.4×)

**Capacity is secondary:**
- Exp 2 → Exp 3: mAP50 0.468 → 0.543 (+16%)
- Largest gains on small classes (bicycle +50%, tricycle +30%)

**Both compound:**
- Exp 1 → Exp 3: mAP50 0.279 → 0.543 (nearly doubled)

See `README.md` for full analysis and `figures/` for comparison plots.